### 슬라이드 분석 모듈

#### 환경 설정

In [25]:
# library import
import os
from dotenv import load_dotenv
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 환경 변수
load_dotenv()
BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")

#### Structured Output 정의

In [26]:
# Slide 모델 정의
class Slide(BaseModel):
    slide_number: int | None = Field(description="슬라이드 번호. 1부터 시작하는 정수. 구분 불가능할 경우 None")
    script: str = Field(description="해당 슬라이드에 포함된 발표 대본")
    keywords: list[str] = Field(description="해당 슬라이드 대본에 포함된 키워드 목록")
    highlights: list[str] = Field(description="해당 슬라이드 대본에서 강조할 부분 인덱스 목록")

# 전체 결과 모델 정의
class SeperatedSlides(BaseModel):
    status: Literal["success", "fail"] = Field(description="슬라이드 구분 성공 여부")
    slides: list[Slide] = Field(description="슬라이드별 발표 대본 목록")

#### System Prompt 작성

In [27]:
SYSTEM_PROMPT = """
너는 발표 대본을 슬라이드별로 분리하는 전문가야.
입력은 하나의 발표 대본 전체 텍스트야.

# 슬라이드 구분 판단 기준
- "Slide", "슬라이드" 또는 숫자같은 명시적 표기만 슬라이드 구분으로 인정해.
- "첫째", "둘째", "다음으로" 같은 서수/전환 표현은 슬라이드 구분이 아니야.
- 대본 전체에서 이런 명시적 구분이 하나도 없거나, 일부 구간에만 있고 나머지 구간은 구분할 수 없으면 반드시 status="fail"로 처리해. 임의로 슬라이드 번호를 만들어내거나 추측해서 나누지 마.

# keywords 공통 규칙 (success/fail 모두 적용, 개수만 다름)
- keywords는 대본에 있는 단어나 어절을 그대로 사용해. 문장 내용을 바탕으로 요약하거나 의미를 추출하지 마.
- keywords의 순서는 대본에 등장하는 순서대로 담아. (중복 단어는 제거해)
- keywords에는 의미 없는 단어, 조사/접속사 등은 포함하지 마.
- 쉼표(,)로 나열된 단어/구는 서로 의미가 다르더라도 무조건 그 중 하나만 keywords에 담아. 나열된 개수만큼 여러 개의 keyword로 쪼개지 마. (예: "발표 시간, 말하기 속도, pause, 목소리 크기, 시선 처리, filler word" -> 이 중 하나만 담기, 나머지는 keywords에서 제외)

# highlights 공통 규칙
- highlights 필드는 빈 배열로 채워. (추후에 강조할 부분을 표시하기 위해 마련한 필드임)

# 전처리 공통 규칙
- 대본 최상단에 제목이 있을 경우 제거해.

# status="success"인 경우
- 대본 전체가 명시적 구분자를 기준으로 빠짐없이 나뉠 수 있을 때만 success로 판단해.
- slides 배열에 모든 슬라이드를 slide_number, script와 함께 순서대로 채워.
- slide_number는 원문에 표기된 번호를 그대로 사용해. (원문에 1, 3, 5만 있으면 그대로 1, 3, 5로 채워.)
- script에는 구분 기호(예: "1.", "Slide 2:") 자체와 구분 기호 뒤에 나오는 소제목은 제거하고, 그 슬라이드에 해당하는 본문 텍스트만 담아.
- keywords에는 해당 슬라이드 대본에서 추출한 중요한 단어 목록을 3~7개 정도 담아.

# status="fail"인 경우
- slide_number은 None으로 채우고, script에는 구분할 수 없는 전체 대본을 그대로 담아.
- keywords는 15개 정도 추출해서 담아.
- 키워드가 등장하는 구간은 편향되지 않게 해줘. (대본의 앞쪽에만 키워드가 몰리지 않도록)
"""


#### LLM 모델 불러오기

In [28]:
os.getenv('OPENAI_MODEL'), os.getenv('OPENAI_BASE_URL')

('openai/gpt-5.6-luna',
 'https://mlapi.run/286e9158-d32e-436d-a23d-36b43fc8e68a/v1')

In [29]:
# LLM 객체 생성
model = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
)

# Structured Output 연결
structured_output = model.with_structured_output(SeperatedSlides)

# 실제 모델 호출이 되는지 간단한 테스트
response = structured_output.invoke("Please separate the following script into slides:\n\n안녕하세요. 오늘은 멸종위기청년에 대해 발표하겠습니다.") 
response

SeperatedSlides(status='success', slides=[Slide(slide_number=1, script='안녕하세요. 오늘은 멸종위기청년에 대해 발표하겠습니다.', keywords=['멸종위기청년', '발표'], highlights=['오늘 발표 주제: 멸종위기청년'])])

#### 테스트용 대본 파일 import

In [30]:
scripts = []

# 테스트용 .txt 파일 읽기
path = os.path.join(os.getcwd(), "대본")
for filename in os.listdir(path):
    if filename.endswith(".txt"):
        with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
            script = file.read()
            scripts.append(script)
            
scripts

['K-pop의 글로벌 성공\n\nSlide 01: K-pop은 어떻게 세계적인 장르가 되었을까?\n\n안녕하세요. 오늘은 K-pop이 한국을 넘어 세계적인 음악 장르로 성장한 이유를 살펴보겠습니다.\n\n몇 년 전까지만 해도 한국 음악은 주로 아시아권을 중심으로 소비되었습니다. 하지만 이제는 미국과 유럽을 비롯한 다양한 지역에서 K-pop을 쉽게 접할 수 있습니다.\n\nSlide 02: 음악과 퍼포먼스\n\nK-pop의 특징 중 하나는 음악과 퍼포먼스를 함께 소비한다는 점입니다.\n\n노래뿐만 아니라 안무와 무대 연출, 뮤직비디오까지 하나의 콘텐츠처럼 만들어집니다.\n\n이러한 시각적인 요소는 언어를 잘 알지 못하는 사람도 K-pop을 쉽게 접하게 만드는 요소가 될 수 있습니다.\n\nSlide 03: 소셜 미디어\n\n소셜 미디어 역시 중요한 역할을 했습니다.\n\n팬들은 짧은 영상이나 공연 영상을 직접 공유하고 다른 사람에게 추천할 수 있습니다.\n\n특히 짧은 영상 콘텐츠는 특정 노래나 안무가 빠르게 확산되는 데 영향을 주었습니다.\n\nSlide 04: 팬덤\n\nK-pop의 팬덤 문화도 빼놓을 수 없습니다.\n\n팬들은 단순히 음악을 듣는 것을 넘어 콘텐츠를 공유하고 공연에 참여하며 서로 소통합니다.\n\n이러한 적극적인 참여가 아티스트와 팬 사이의 관계를 강화합니다.\n\nSlide 05: 앞으로의 과제\n\n하지만 글로벌 시장에서 계속 성장하기 위해서는 해결해야 할 문제도 있습니다.\n\n아티스트의 활동 지속 가능성이나 지나치게 빠른 콘텐츠 생산 구조 등에 대한 고민이 필요합니다.\n\nSlide 06: 결론\n\nK-pop의 글로벌 성공은 음악 하나의 힘만으로 설명하기 어렵습니다.\n\n음악과 퍼포먼스, 소셜 미디어, 팬덤 문화가 서로 결합하면서 만들어진 결과라고 볼 수 있습니다.\n\n감사합니다.',
 '3박 4일 제주도 여행 계획\n\n1\n\n안녕하세요. 지금부터 3박 4일 제주도 여행 계획을 소개하겠습니다.\n\n이번 여행의 목표는 유

#### 전처리 함수 정의 (LLM 응답 후처리용)

In [31]:
import re

def clean_script(text: str) -> str:
    # 헤더(#, ##, ...) 제거
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    # 굵게/기울임 강조 기호 제거: **text**, __text__, *text*, _text_
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    text = re.sub(r"__(.+?)__", r"\1", text)
    text = re.sub(r"(?<!\w)\*(.+?)\*(?!\w)", r"\1", text)
    text = re.sub(r"(?<!\w)_(.+?)_(?!\w)", r"\1", text)
    # 인라인 코드 백틱(`code`) 제거
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # 마크다운 링크 [텍스트](URL) -> 텍스트 (슬라이드 마커 [슬라이드 1] 등은 뒤에 괄호가 없어 영향 없음)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # 개행 및 연속 공백을 단일 공백으로 정리
    text = re.sub(r"\s+", " ", text)
    return text.strip()


#### 단일 대본 test

In [36]:
result = structured_output.invoke(
    [
        ("system", SYSTEM_PROMPT),
        ("user", scripts[8])
    ]
)
data = result.model_dump()
data


{'status': 'success',
 'slides': [{'slide_number': 1,
   'script': '안녕하세요. 오늘은 전기차의 미래에 대해 발표하겠습니다.\n\n최근 도로에서 전기차를 쉽게 찾아볼 수 있습니다. 자동차 산업에서도 전기차의 비중이 점점 커지고 있습니다.',
   'keywords': ['전기차', '미래', '도로', '자동차 산업', '비중'],
   'highlights': []},
  {'slide_number': 2,
   'script': '전기차가 주목받는 이유는 무엇일까요?\n\n가장 큰 이유 중 하나는 배기가스가 없다는 점입니다. 또한 기존 내연기관 자동차와 비교했을 때 구조가 단순하다는 장점도 있습니다.',
   'keywords': ['전기차', '배기가스', '내연기관 자동차', '구조', '장점'],
   'highlights': []},
  {'slide_number': 3,
   'script': '하지만 해결해야 할 문제도 있습니다.\n\n대표적인 문제가 충전 인프라입니다. 장거리 여행을 할 때 충전소를 찾는 것이 불편할 수 있고 충전에 필요한 시간도 고려해야 합니다.\n\n배터리 가격과 수명 역시 중요한 문제입니다.',
   'keywords': ['충전 인프라', '장거리 여행', '충전소', '충전', '배터리 가격', '수명'],
   'highlights': []},
  {'slide_number': 4,
   'script': '앞으로는 배터리 기술이 전기차의 경쟁력을 결정하는 중요한 요소가 될 것입니다.\n\n더 많은 에너지를 저장하면서도 가격은 낮추고 충전 시간은 줄이는 기술이 필요합니다.',
   'keywords': ['배터리 기술', '전기차', '경쟁력', '에너지', '가격', '충전 시간'],
   'highlights': []},
  {'slide_number': 5,
   'script': '전기차가 내연기관 자동차를 단기간에 완전히 대체할 것이라고 단정하기는 어렵습니

In [37]:
for slide in data["slides"]:
    slide["script"] = clean_script(slide["script"])
data

{'status': 'success',
 'slides': [{'slide_number': 1,
   'script': '안녕하세요. 오늘은 전기차의 미래에 대해 발표하겠습니다. 최근 도로에서 전기차를 쉽게 찾아볼 수 있습니다. 자동차 산업에서도 전기차의 비중이 점점 커지고 있습니다.',
   'keywords': ['전기차', '미래', '도로', '자동차 산업', '비중'],
   'highlights': []},
  {'slide_number': 2,
   'script': '전기차가 주목받는 이유는 무엇일까요? 가장 큰 이유 중 하나는 배기가스가 없다는 점입니다. 또한 기존 내연기관 자동차와 비교했을 때 구조가 단순하다는 장점도 있습니다.',
   'keywords': ['전기차', '배기가스', '내연기관 자동차', '구조', '장점'],
   'highlights': []},
  {'slide_number': 3,
   'script': '하지만 해결해야 할 문제도 있습니다. 대표적인 문제가 충전 인프라입니다. 장거리 여행을 할 때 충전소를 찾는 것이 불편할 수 있고 충전에 필요한 시간도 고려해야 합니다. 배터리 가격과 수명 역시 중요한 문제입니다.',
   'keywords': ['충전 인프라', '장거리 여행', '충전소', '충전', '배터리 가격', '수명'],
   'highlights': []},
  {'slide_number': 4,
   'script': '앞으로는 배터리 기술이 전기차의 경쟁력을 결정하는 중요한 요소가 될 것입니다. 더 많은 에너지를 저장하면서도 가격은 낮추고 충전 시간은 줄이는 기술이 필요합니다.',
   'keywords': ['배터리 기술', '전기차', '경쟁력', '에너지', '가격', '충전 시간'],
   'highlights': []},
  {'slide_number': 5,
   'script': '전기차가 내연기관 자동차를 단기간에 완전히 대체할 것이라고 단정하기는 어렵습니다. 하지만 충전 인프라와 

#### highlight 구간 추출

In [ ]:
for slide in data["slides"]:
    for keyword in slide["keywords"]:
        if keyword not in slide["script"]:
            print(f"Keyword '{keyword}' not found in slide {slide['slide_number']}.")
        start = slide["script"].find(keyword)
        end = start + len(keyword)
        slide["highlights"].append(f"{start}:{end}")
data

{'status': 'success',
 'slides': [{'slide_number': 1,
   'script': '안녕하세요. 오늘은 전기차의 미래에 대해 발표하겠습니다. 최근 도로에서 전기차를 쉽게 찾아볼 수 있습니다. 자동차 산업에서도 전기차의 비중이 점점 커지고 있습니다.',
   'keywords': ['전기차', '미래', '도로', '자동차 산업', '비중'],
   'highlights': ['11:14', '16:18', '35:37', '60:66', '75:77']},
  {'slide_number': 2,
   'script': '전기차가 주목받는 이유는 무엇일까요? 가장 큰 이유 중 하나는 배기가스가 없다는 점입니다. 또한 기존 내연기관 자동차와 비교했을 때 구조가 단순하다는 장점도 있습니다.',
   'keywords': ['전기차', '배기가스', '내연기관 자동차', '구조', '장점'],
   'highlights': ['0:3', '35:39', '57:65', '74:76', '84:86']},
  {'slide_number': 3,
   'script': '하지만 해결해야 할 문제도 있습니다. 대표적인 문제가 충전 인프라입니다. 장거리 여행을 할 때 충전소를 찾는 것이 불편할 수 있고 충전에 필요한 시간도 고려해야 합니다. 배터리 가격과 수명 역시 중요한 문제입니다.',
   'keywords': ['충전 인프라', '장거리 여행', '충전소', '충전', '배터리 가격', '수명'],
   'highlights': ['30:36', '41:47', '53:56', '30:32', '95:101', '103:105']},
  {'slide_number': 4,
   'script': '앞으로는 배터리 기술이 전기차의 경쟁력을 결정하는 중요한 요소가 될 것입니다. 더 많은 에너지를 저장하면서도 가격은 낮추고 충전 시간은 줄이는 기술이 필요합니다.',
   'keywords': ['배터리 기술', '전기차', '경쟁력', 

#### 전체 대본 test

In [ ]:
# results = []

# for script in scripts:
#     result = structured_output.invoke(
#         [
#             ("system", SYSTEM_PROMPT),
#             ("user", script)
#         ]
#     )
#     data = result.model_dump()
#     for slide in data["slides"]:
#         slide["script"] = clean_script(slide["script"])
#     results.append(data)

# results


[{'status': 'success',
  'slides': [{'slide_number': 1,
    'script': '안녕하세요. 오늘은 K-pop이 한국을 넘어 세계적인 음악 장르로 성장한 이유를 살펴보겠습니다. 몇 년 전까지만 해도 한국 음악은 주로 아시아권을 중심으로 소비되었습니다. 하지만 이제는 미국과 유럽을 비롯한 다양한 지역에서 K-pop을 쉽게 접할 수 있습니다.',
    'keywords': ['K-pop', '한국', '세계적인 음악 장르', '아시아권', '미국과 유럽'],
    'highlights': []},
   {'slide_number': 2,
    'script': 'K-pop의 특징 중 하나는 음악과 퍼포먼스를 함께 소비한다는 점입니다. 노래뿐만 아니라 안무와 무대 연출, 뮤직비디오까지 하나의 콘텐츠처럼 만들어집니다. 이러한 시각적인 요소는 언어를 잘 알지 못하는 사람도 K-pop을 쉽게 접하게 만드는 요소가 될 수 있습니다.',
    'keywords': ['K-pop', '음악과 퍼포먼스', '안무', '무대 연출', '뮤직비디오', '시각적인 요소'],
    'highlights': []},
   {'slide_number': 3,
    'script': '소셜 미디어 역시 중요한 역할을 했습니다. 팬들은 짧은 영상이나 공연 영상을 직접 공유하고 다른 사람에게 추천할 수 있습니다. 특히 짧은 영상 콘텐츠는 특정 노래나 안무가 빠르게 확산되는 데 영향을 주었습니다.',
    'keywords': ['소셜 미디어', '팬들', '짧은 영상', '공연 영상', '공유', '영상 콘텐츠', '확산'],
    'highlights': []},
   {'slide_number': 4,
    'script': 'K-pop의 팬덤 문화도 빼놓을 수 없습니다. 팬들은 단순히 음악을 듣는 것을 넘어 콘텐츠를 공유하고 공연에 참여하며 서로 소통합니다. 이러한 적극적인 참여가 아티스트와 팬 사이의 관계를 강화합니다.',
